# Reinforcement-learning trader search on a Kaggle GPU

Runs configurations of the pre-registered search
(`docs/specs/2026-11-11-rl-daytrader-design.md`) with `hft.ml.rl_train`. Attach the private
datasets `hft-ml-data` and `hft-ml-code`, set **Accelerator: GPU T4**, set `CONFIGS` below and
**Run all**. Each finished configuration is written to `/kaggle/working/rl/<id>/` and zipped
into `rl-results.zip`. Validation years only; the test years are never loaded.

In [ ]:
import sys, json, subprocess, zipfile, time
from pathlib import Path

inputs, work = Path("/kaggle/input"), Path("/kaggle/working")


def find(pattern):
    return next(iter(inputs.rglob(pattern)), None) or next(iter((work / "unpacked").rglob(pattern)), None)


# Datasets uploaded as zip files may arrive still zipped: unpack them once.
if find("calendar.json") is None or find("hft/ml/rl_train.py") is None:
    for archive in inputs.rglob("*.zip"):
        with zipfile.ZipFile(archive) as z:
            z.extractall(work / "unpacked")
code_dir = find("hft/ml/rl_train.py").parent.parent.parent
data_dir = find("calendar.json").parent
sys.path.insert(0, str(code_dir))
import torch
print("code:", (code_dir / "COMMIT").read_text().strip() if (code_dir / "COMMIT").exists() else "?")
print("data:", data_dir, "GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
from hft.ml.rl_train import grid, run_config
from hft.ml.train import load_inputs

CONFIGS = []  # configuration ids; empty runs all 16 (more than one 12-hour session)
started = time.time()
inputs = load_inputs(data_dir)
print(f"inputs loaded in {time.time() - started:.0f}s")
for config in grid():
    if CONFIGS and config["id"] not in CONFIGS:
        continue
    t0 = time.time()
    run_config(config, inputs, work / "rl", "cuda")
    print(f"{config['id']} took {time.time() - t0:.0f}s", flush=True)
subprocess.run(["zip", "-qr", str(work / "rl-results.zip"), "rl"], cwd=work, check=True)
print("download rl-results.zip")